In [4]:
# !pip install nba_api

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings

warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


def fetch_team_data(seasons=None):
    """Fetch NBA team data from multiple seasons."""
    if seasons is None:
        seasons = [
            '2019-20',
            '2020-21',
            '2021-22',
            '2022-23',
            '2023-24',
            '2024-25'
        ]

    print(f"Fetching NBA team data for {len(seasons)} seasons...")

    all_dfs = []

    for season in seasons:
        print(f"  -> Fetching {season}")
        team_stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        )
        df = team_stats.get_data_frames()[0]
        df['SEASON'] = season  # 👈 track season
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Total rows collected: {len(combined_df)}")

    return combined_df

def prepare_data(df):
    """Prepare features and create target variable."""
    features = [
        'OFF_RATING',
        'DEF_RATING',
        'NET_RATING',
        'PACE',
        'TS_PCT',
        'EFG_PCT',
        'AST_RATIO',
        'REB_PCT',
        'TM_TOV_PCT',
        'PIE'
    ]

    df_clean = df[['TEAM_NAME'] + features + ['W_PCT']].dropna()
    df_clean['WIN_TIER'] = pd.qcut(
        df_clean['W_PCT'],
        3,
        labels=['Low', 'Mid', 'High'],
        duplicates='drop'
    )

    X = df_clean[features].copy()
    y = df_clean['WIN_TIER']
    team_names = df_clean['TEAM_NAME']

    print(f"Data shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts().sort_index()}")

    return X, y, team_names


def plot_labeled_confusion_matrix(y_true, y_pred, class_names, title, filename):
    """Plot and save a labeled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Greens',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def plot_side_by_side_confusion_matrices(y_true, y_pred_rf, y_pred_bag,
                                          class_names, filename):
    """Plot and save side-by-side confusion matrices for comparison."""
    cm_rf = confusion_matrix(y_true, y_pred_rf, labels=class_names)
    cm_bag = confusion_matrix(y_true, y_pred_bag, labels=class_names)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Random Forest
    sns.heatmap(
        cm_rf,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0],
        cbar_kws={'shrink': 0.8}
    )
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title('Random Forest - Confusion Matrix')

    # Bagging
    sns.heatmap(
        cm_bag,
        annot=True,
        fmt='d',
        cmap='Greens',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1],
        cbar_kws={'shrink': 0.8}
    )
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title('Bagging - Confusion Matrix')

    plt.suptitle('Ensemble Comparison: Random Forest vs Bagging', fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def main():
    df = fetch_team_data()
    X, y, team_names = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    class_names = y.unique().tolist()

    # ========== Random Forest ==========
    print("\n=== Random Forest ===")
    rf_clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
    rf_clf.fit(X_train, y_train)
    y_pred_rf = rf_clf.predict(X_test)

    rf_acc = accuracy_score(y_test, y_pred_rf)
    print(f"Accuracy: {rf_acc:.4f}")
    print(classification_report(y_test, y_pred_rf, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred_rf, class_names,
        f'Random Forest - Confusion Matrix\nAccuracy: {rf_acc:.4f}',
        'outputs/ensemble_random_forest_confusion.png'
    )

    print("\nFeature Importance (Random Forest):")
    for feat, imp in zip(X.columns, rf_clf.feature_importances_):
        print(f"  {feat}: {imp:.4f}")

    # ========== Bagging Classifier ==========
    print("\n=== Bagging Classifier ===")
    base_tree = DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    )
    bag_clf = BaggingClassifier(
        estimator=base_tree,
        n_estimators=100,
        max_samples=0.8,
        max_features=0.8,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )
    bag_clf.fit(X_train, y_train)
    y_pred_bag = bag_clf.predict(X_test)

    bag_acc = accuracy_score(y_test, y_pred_bag)
    print(f"Accuracy: {bag_acc:.4f}")
    print(classification_report(y_test, y_pred_bag, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred_bag, class_names,
        f'Bagging - Confusion Matrix\nAccuracy: {bag_acc:.4f}',
        'outputs/ensemble_bagging_confusion.png'
    )

    # ========== Comparison ==========
    print("\n" + "="*60)
    print("COMPARISON: Random Forest vs Bagging")
    print("="*60)
    print(f"\nRandom Forest Accuracy: {rf_acc:.4f}")
    print(f"Bagging Accuracy:       {bag_acc:.4f}")

    if rf_acc > bag_acc:
        diff = rf_acc - bag_acc
        print(f"\nRandom Forest outperforms Bagging by {diff:.4f} ({diff*100:.2f}%)")
    else:
        diff = bag_acc - rf_acc
        print(f"\nBagging outperforms Random Forest by {diff:.4f} ({diff*100:.2f}%)")

    plot_side_by_side_confusion_matrices(
        y_test, y_pred_rf, y_pred_bag,
        class_names,
        'outputs/ensemble_comparison.png'
    )


if __name__ == "__main__":
    main()


Fetching NBA team data for 6 seasons...
  -> Fetching 2019-20
  -> Fetching 2020-21
  -> Fetching 2021-22
  -> Fetching 2022-23
  -> Fetching 2023-24
  -> Fetching 2024-25
Total rows collected: 180
Data shape: (180, 10)
Target distribution:
WIN_TIER
Low     60
Mid     66
High    54
Name: count, dtype: int64

=== Random Forest ===
Accuracy: 0.9167
              precision    recall  f1-score   support

         Low       1.00      0.91      0.95        11
        High       1.00      0.83      0.91        12
         Mid       0.81      1.00      0.90        13

    accuracy                           0.92        36
   macro avg       0.94      0.91      0.92        36
weighted avg       0.93      0.92      0.92        36

Saved: outputs/ensemble_random_forest_confusion.png

Feature Importance (Random Forest):
  OFF_RATING: 0.0702
  DEF_RATING: 0.0917
  NET_RATING: 0.3009
  PACE: 0.0256
  TS_PCT: 0.0561
  EFG_PCT: 0.0413
  AST_RATIO: 0.0204
  REB_PCT: 0.0497
  TM_TOV_PCT: 0.0307
  PIE: 0.